# Resolving Conflicts by Encoding the Precedences into the Grammar

The notebook [01-Conflicts.ipynb](01-Conflicts.ipynb) has shown that the following grammar is *ambiguous*,
because it does not specify the precedences of the arithmetical operators:
```
    expr : expr "+" expr
         | expr "-" expr
         | expr "*" expr
         | expr "/" expr
         | expr "^" expr
         | "(" expr ")"
         | NUMBER
```

The parser generators of the `yacc` family, among them `Ply` and `Menhir`, resolve the resulting *shift/reduce
conflicts* with *operator precedence declarations*.  In `Menhir`, they have the form
```
    %left  PLUS MINUS     /* precedence 1 */
    %left  STAR SLASH     /* precedence 2 */
    %right CIRCUMFLEX     /* precedence 3 */
```

The Python version of this notebook uses `Lark`, and <b style="color:red">`Lark` does not support operator precedence
declarations.</b>  There is no way to attach a precedence or an associativity to a terminal, and there is no way to
influence the resolution of an individual shift/reduce conflict: `Lark` resolves *every* shift/reduce conflict in
favour of shifting.

Therefore, the Python version encodes the precedences and associativities into the grammar itself, and so do we.
The notebook `../Chapter-13/Compiler-For.ipynb` shows how precedence declarations are used.

## Encoding the Precedences

The recipe is the one that we have already used in the chapter on context-free languages:  we introduce one
syntactical variable $v_n$ for every precedence level $n$ and define the variable of a given precedence level
in terms of the variable of the next higher precedence level.  Let $o$ be an operator of precedence level $n$.

* If $o$ is **left associative**, the corresponding rule is **left recursive**:
  $$ v_n \rightarrow v_n \; o \; v_{n+1} $$
* If $o$ is **right associative**, the rule is **right recursive**:
  $$ v_n \rightarrow v_{n+1} \; o \; v_n $$
* If $o$ is **non-associative**, the rule uses neither recursion:
  $$ v_n \rightarrow v_{n+1} \; o \; v_{n+1} $$

For the grammar above we need three precedence levels plus one variable for the expressions that bind most
tightly:

| variable | precedence | operators | associativity |
|:---------|:-----------|:----------|:--------------|
| `expr`   | 1          | `+`, `-`  | left          |
| `prod`   | 2          | `*`, `/`  | left          |
| `fact`   | 3          | `^`       | right         |
| `atom`   | &mdash;    | &mdash;   | &mdash;       |

Note how the associativities show up in the grammar below:  the rules for `expr` and `prod` are *left*
recursive, while the rule `fact : atom "^" fact` is *right* recursive.

Two features of `Lark` are used in the Python version to keep the parse tree readable.  In `Menhir`, the semantic
actions take their place.

1. If the name of a syntactical variable is prefixed with a question mark, as in `?expr`, then `Lark` removes a node of
   this variable from the tree whenever it has exactly one child.  In `Menhir`, the action of a chain rule like
   `expr : prod` simply returns the tree of its child: `{ p }`.  Without this, every number would be buried under a
   chain `expr` &rarr; `prod` &rarr; `fact` &rarr; `atom` that carries no information.
2. In `Lark`, an alternative can be given an *alias* using the operator `->`.  The alias becomes the name of the node
   that is created and, later on, the name of the method of the `Transformer`.  In `Menhir`, the action
   `{ Tree ("add", [e; p]) }` creates a node with the name `add`.

The alternative `NUMBER -> number` creates a node, although it has a single child.  The rule for parenthesized
expressions returns the tree of the expression in parentheses.

Neither the actions nor the features of `Lark` have any influence on the parse table:  they can neither introduce
nor remove a conflict.

In [ ]:
let lexer = {|
{
  open Expr_parser
}

rule token = parse
  | [' ' '\t' '\r' '\n']+        { token lexbuf }      (* %ignore WS *)
  | '+'                          { PLUS }
  | '-'                          { MINUS }
  | '*'                          { STAR }
  | '/'                          { SLASH }
  | '^'                          { CIRCUMFLEX }
  | '('                          { LPAR }
  | ')'                          { RPAR }
  | '0' | ['1'-'9'] ['0'-'9']*   { NUMBER (Parse_tree.token "NUMBER" lexbuf) }
  | eof                          { EOF }
  | _                            { raise (Parse_tree.unexpected lexbuf ("No terminal matches " ^ repr (Lexing.lexeme lexbuf))) }
|}

let grammar = {|
%{
  open Parse_tree
%}

%token <Parse_tree.tok> NUMBER
%token PLUS MINUS STAR SLASH CIRCUMFLEX LPAR RPAR EOF

%start <Parse_tree.tree> start
%type  <Parse_tree.tree> expr prod fact atom

%%

start: e = expr EOF                    { e }

expr : e = expr PLUS p = prod          { Tree ("add", [e; p]) }
     | e = expr MINUS p = prod         { Tree ("sub", [e; p]) }
     | p = prod                        { p }

prod : p = prod STAR f = fact          { Tree ("mul", [p; f]) }
     | p = prod SLASH f = fact         { Tree ("div", [p; f]) }
     | f = fact                        { f }

fact : a = atom CIRCUMFLEX f = fact    { Tree ("power", [a; f]) }
     | a = atom                        { a }

atom : LPAR e = expr RPAR              { e }
     | n = NUMBER                      { Tree ("number", [Token n]) }
|}

## Specification of the Parser

We create the parser with the option `--strict`.  In this mode, `Menhir` refuses to create a parser if it finds a
conflict and `generate_parser_with` raises the exception `Grammar_error`, instead of silently resolving the conflict in
favour of shifting.  This corresponds to the keyword argument `strict=True` of `Lark`.  It is a good idea to develop a
grammar this way, for then a conflict cannot be overlooked.

Since no exception is raised below and no warning is printed, our grammar is an `LALR(1)` grammar.

In [ ]:
#use "../lib/nb.ml";;
load_lib "parser_generator.ml";;
open Parse_tree
;;
generate_parser_with ~options:"--lalr --strict" ~name:"expr" ~lexer ~grammar

## Inspecting the LALR States

`Ply` writes the LALR states into the file `parser.out`.  `Menhir` writes them into the file
`_build/expr_parser.automaton.resolved`, since `generate_parser_with` passes the option `--dump-resolved`.  The
function `dump_states` is the same one that we have used in [01-Conflicts.ipynb](01-Conflicts.ipynb): it prints this
file.

Remember that `Menhir` only shows the *kernel* of every state, i.e. those marked rules where the marker is not at the
beginning.

In [ ]:
let dump_states (name : string) : unit =
  print_string (read_file (Printf.sprintf "_build/%s_parser.automaton.resolved" name));
  flush stdout

The Python version has to build the parser a second time with `debug=True` in order to look at the states.  This is
not necessary with `Menhir`, since the file has already been written.  `Menhir` marks the position of the parser
inside a rule with the character `.` instead of the bullet `•` that we use in the lecture notes, and the end of the
input is the token `EOF`.

Note that there is no state that offers both a `shift` and a `reduce` action for the same token:  all
conflicts are gone.

In [ ]:
dump_states "expr"

## Building an Abstract Syntax Tree

The parse tree returned by the parser has the type `Parse_tree.tree`.  As in [01-Conflicts.ipynb](01-Conflicts.ipynb),
the function `ast_builder` converts it into an abstract syntax tree of the type `expr`, and the function `to_nested`
converts an abstract syntax tree into a nested tuple, so that we can reuse the function `tuple2dot`.  For every name of a
node occurring in the grammar, `ast_builder` has a case of the same name, like the `Transformer` of the Python version
has a method of the same name.

The chain rules and the rule for parenthesized expressions need no case:  they do not create nodes and therefore never
reach `ast_builder`.

In [ ]:
run_notebook "../AST2Dot.ipynb"

In [ ]:
type expr =
  | Number of int
  | Add    of expr * expr
  | Sub    of expr * expr
  | Mul    of expr * expr
  | Div    of expr * expr
  | Power  of expr * expr

let rec ast_builder (t : tree) : expr =
  match t with
  | Tree ("add", [ lhs; rhs ]) -> Add (ast_builder lhs, ast_builder rhs)
  | Tree ("sub", [ lhs; rhs ]) -> Sub (ast_builder lhs, ast_builder rhs)
  | Tree ("mul", [ lhs; rhs ]) -> Mul (ast_builder lhs, ast_builder rhs)
  | Tree ("div", [ lhs; rhs ]) -> Div (ast_builder lhs, ast_builder rhs)
  | Tree ("power", [ lhs; rhs ]) -> Power (ast_builder lhs, ast_builder rhs)
  | Tree ("number", [ Token n ]) -> Number (int_of_string n.value)
  | _ -> invalid_arg ("ast_builder: unexpected node " ^ pretty t)

let rec to_nested (e : expr) : nested =
  let node op lhs rhs = Tuple (op, [ to_nested lhs; to_nested rhs ]) in
  match e with
  | Number n -> Int n
  | Add (lhs, rhs) -> node "+" lhs rhs
  | Sub (lhs, rhs) -> node "-" lhs rhs
  | Mul (lhs, rhs) -> node "*" lhs rhs
  | Div (lhs, rhs) -> node "/" lhs rhs
  | Power (lhs, rhs) -> node "^" lhs rhs

The function `test s` takes a string `s` as its argument and tries to parse this string.  If all goes well, an abstract
syntax tree is displayed and returned as `Some t`.  If the string can't be parsed, the parser raises the exception
`Unexpected_input`.  This exception carries the message, the line, the column, and the position of the error, so there
is no need for a function like `p_error`.  The function `get_context` shows the line containing the error together with a
marker.

In [ ]:
let test (s : string) : expr option =
  match parse_string Expr_parser.start Expr_lexer.token s with
  | exception Unexpected_input { message; line; column; position } ->
      Printf.printf "Syntax error:\n%s at line %d, column %d.\n\n%s%!" message line column (get_context s position);
      None
  | parse_tree ->
      let t = ast_builder parse_tree in
      let d = tuple2dot (to_nested t) in
      Graphviz.display d;
      Some t

In [ ]:
test "2^3^4*5+6-7/8^9"

In [ ]:
test "1+2*3^4"

In [ ]:
test "1 * 2 + (3^4)^5"

The next two examples check the associativities:  the additive operators group to the left, while the
exponentiation operator groups to the right.

In [ ]:
test "1-2-3"

In [ ]:
test "2^3^4"

Finally, an example of a string that cannot be parsed.

In [ ]:
test "1 + * 2"